# Querycraft v2 notebook C: 4 bit AWQ (W4A16) of the merged 7B DoRA model

Settings: **GPU T4 x2** (or Colab T4 as a backup), **Internet on**. Same three inputs. Quantizes with llm-compressor's
AWQ recipe, calibrated on 64 val prompts (up to 1,024 tokens) with their gold SQL, as for the 1.5B model (val is the split choices are made on; dev and held
back are never touched). Output: `/kaggle/working/out/qwen7b_dora_awq.tar.gz` (~5.5 GB), which the laptop benchmark
loads with vLLM (`make bench`). Estimated 1 GPU hour.


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
def sh(cmd):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True)
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
IN = Path("/kaggle/input")
def find(slug):
    hits = [p for p in IN.glob("*") if slug in p.name] + [p for p in IN.glob("*/*/*") if slug in str(p)]
    assert hits, f"attach the dataset {slug} (Add Input)"; return hits[0]
W, T = Path("/kaggle/working"), Path("/kaggle/tmp")
QC = T / "qc"; OUT = W / "out"  # code + data in tmp (not saved); only out/ is kept
OUT.mkdir(exist_ok=True); T.mkdir(exist_ok=True)
def unpack(src, dst):
    """Kaggle may keep uploaded folders as .zip files: extract those, copy the rest."""
    dst.mkdir(parents=True, exist_ok=True)
    for p in Path(src).iterdir():
        if p.suffix == ".zip":
            zipfile.ZipFile(p).extractall(dst)
        elif p.is_dir():
            shutil.copytree(p, dst / p.name, dirs_exist_ok=True)
        else:
            shutil.copy(p, dst / p.name)
unpack(find("querycraft-code"), QC)
# Kaggle drops the top folder of a single uploaded zip: BIRD arrives as raw/ + processed/, the adapter as bare files
unpack(find("querycraft-bird-eval"), QC / "data")
if (QC / "data" / "data").is_dir():                 # in case a future upload keeps the folder
    shutil.copytree(QC / "data" / "data", QC / "data", dirs_exist_ok=True)
CKPT = T / "runs/qwen7b_dora_best/ckpt/checkpoint-560"
unpack(find("querycraft-7b-dora"), CKPT)
if (CKPT / "checkpoint-560").is_dir():
    shutil.copytree(CKPT / "checkpoint-560", CKPT, dirs_exist_ok=True)
assert (CKPT / "adapter_config.json").exists() and (QC / "data/processed/dev.jsonl").exists(), "unexpected dataset layout"
os.chdir(QC); os.environ["PYTHONPATH"] = str(QC); sys.path.insert(0, str(QC))
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
(QC / "runs").mkdir(exist_ok=True)
print(sorted(p.name for p in QC.iterdir()))


In [ ]:
# An isolated Python 3.12 venv with the laptop's exact stack (vLLM 0.32, torch 2.13 cu129). Kaggle's preinstalled
# packages (torchaudio, torchao, ...) are built for its own torch and break transformers/peft when mixed in.
VENV = T / "venv"; VPY = str(VENV / "bin/python")
sh("pip install -q uv")
sh(f"uv venv -q {VENV} --python 3.12")
sh(f"uv pip install --python {VPY} -q --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu129 "
   "'torch==2.13.0+cu129' 'vllm @ https://github.com/vllm-project/vllm/releases/download/v0.32.0/vllm-0.32.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl' "
   "'transformers>=5.16.1,<5.20.0' peft accelerate sqlglot pyyaml opentelemetry-api httpx")
sh(f"uv pip uninstall --python {VPY} torchcodec || true")


In [ ]:
# DoRA is not servable by vLLM: merge it into the 7B base on CPU (bf16 weights, ~15 GB in /kaggle/tmp)
t0 = time.time()
sh(f"{VPY} -c \"from querycraft.infer import merge_adapter; print(merge_adapter('Qwen/Qwen2.5-Coder-7B-Instruct', '{CKPT}'))\"")
MERGED = str(CKPT.parent.parent / "merged" / CKPT.name)
print(MERGED, f"{time.time() - t0:.0f} s")


In [ ]:
QVPY = str(T / "qvenv/bin/python")
sh(f"uv venv -q {T / 'qvenv'} --python 3.12")
sh(f"uv pip install --python {QVPY} -q --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu129 "
   "'torch==2.13.0+cu129' llmcompressor 'transformers>=5.16.1,<5.20.0' datasets accelerate sqlglot pyyaml")
QUANT = """
import json, sys, time, torch
sys.path.insert(0, '%s')
from datasets import Dataset
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from transformers import AutoModelForCausalLM, AutoTokenizer
from querycraft.data import load_split
from querycraft.prompts import PromptBuilder
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-Coder-7B-Instruct')
pb = PromptBuilder(tok, 1280)
rows = load_split('val')[:64]
texts = [tok.apply_chat_template(pb.messages(r) + [{'role': 'assistant', 'content': r['sql']}], tokenize=False) for r in rows]
ds = Dataset.from_dict({'text': texts}).map(lambda x: tok(x['text'], truncation=True, max_length=1024), remove_columns=['text'])
model = AutoModelForCausalLM.from_pretrained('%s', dtype=torch.float16, device_map='auto')
t0 = time.time()
oneshot(model=model, dataset=ds, recipe=[AWQModifier(targets=['Linear'], scheme='W4A16_ASYM', ignore=['lm_head'])],
        max_seq_length=1024, num_calibration_samples=len(texts))
model.save_pretrained('%s', save_compressed=True); tok.save_pretrained('%s')
json.dump({'calibration': '64 val prompts + gold SQL, 1024 tokens', 'scheme': 'W4A16_ASYM', 'seconds': round(time.time() - t0)},
          open('%s/querycraft_quant.json', 'w'))
""" % (QC, MERGED, T / "qwen7b_dora_awq", T / "qwen7b_dora_awq", T / "qwen7b_dora_awq")
(T / "quant.py").write_text(QUANT)
sh(f"cd {QC} && {QVPY} {T / 'quant.py'}")
sh(f"cd {T} && tar czf {OUT}/qwen7b_dora_awq.tar.gz qwen7b_dora_awq")
